# Analyzer-Based Extraction with Page Detection

Two-step pipeline:
1. **Scout** — run `prebuilt-layout` (~15 s per PDF) to find which pages contain financial statements.
2. **Extract** — run the custom analyzer **only on those pages** via `content_range`, avoiding the ~25 min full-document cost.

Handles multi-page tables and sections split across multiple physical tables (e.g. Equity statements that span two pages with separate period tables).

In [ ]:
import json, os, re, time
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.ai.contentunderstanding import ContentUnderstandingClient

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(REPO_ROOT / ".env")

endpoint = os.environ["FOUNDRY_ENDPOINT"].split("/api/projects/")[0].rstrip("/") + "/"
credential = ClientSecretCredential(
    tenant_id=os.environ["AZURE_TENANT_ID"],
    client_id=os.environ["AZURE_CLIENT_ID"],
    client_secret=os.environ["AZURE_CLIENT_SECRET"],
)
client = ContentUnderstandingClient(endpoint=endpoint, credential=credential, api_version="2025-11-01")
print("client ready:", endpoint)

In [ ]:
# Deploy (or update) the custom analyzer
ANALYZER_ID = "sec_financial_tables_v1"
template_path = REPO_ROOT / "analyzers" / f"{ANALYZER_ID}.json"

tmpl = json.loads(template_path.read_text(encoding="utf-8"))
tmpl["models"] = {
    "completion": os.environ["GPT41_MODEL_DEPLOYMENT"],
    "embedding": os.environ["EMBEDDING_MODEL_DEPLOYMENT"],
}

t0 = time.time()
client.begin_create_analyzer(analyzer_id=ANALYZER_ID, resource=tmpl, allow_replace=True).result()
print(f"Analyzer '{ANALYZER_ID}' deployed in {time.time() - t0:.1f}s")

In [ ]:
# Prebuilt-layout scan — identify tables and pages in all PDFs
pdf_dir = REPO_ROOT / "email" / "attachements"
pdfs = sorted(pdf_dir.glob("*.pdf"))
print(f"Found {len(pdfs)} PDFs: {[p.name for p in pdfs]}\n")

layout_results = {}
for pdf in pdfs:
    t0 = time.time()
    poller = client.begin_analyze_binary(
        analyzer_id="prebuilt-layout",
        binary_input=pdf.read_bytes(),
        content_type="application/pdf",
    )
    layout_results[pdf.stem] = poller.result().as_dict()
    elapsed = time.time() - t0
    n_tables = len(layout_results[pdf.stem]["contents"][0].get("tables", []))
    n_pages = layout_results[pdf.stem]["contents"][0].get("endPageNumber", "?")
    print(f"  {pdf.name}: {n_pages} pages, {n_tables} tables, {elapsed:.1f}s")

In [ ]:
# --- Page detection ---
#
# Strategy: find the contiguous page range that contains the financial statements,
# then build per-statement page groups for focused extraction.
#
# 1. Scan paragraphs for FS title patterns → collect (page, statement_type).
#    Filter out long narrative paragraphs (> 150 chars) that mention FS names in passing.
# 2. The FS section is a contiguous block.  Separate TOC hits (isolated early pages)
#    from the actual statement cluster (where multiple types appear close together).
# 3. Extend for multi-page tables: if a table starts inside the FS range and its
#    cells span beyond, include those extra pages.
# 4. Build per-statement page groups — each statement gets its own page range for
#    focused extraction (avoids the LLM processing all statements at once).
#    TOC/index pages (those hitting multiple statement types) are excluded.

_FS_TITLE = [
    (re.compile(r"consolidated\s+balance\s+sheets?", re.I), "BalanceSheet"),
    (re.compile(r"consolidated\s+statements?\s+of\s+(operations|income|earnings)", re.I), "IncomeStatement"),
    (re.compile(r"consolidated\s+statements?\s+of\s+comprehensive\s+(income|loss)", re.I), "ComprehensiveIncome"),
    (re.compile(r"consolidated\s+statements?\s+of.{0,20}equity", re.I), "Equity"),
    (re.compile(r"consolidated\s+statements?\s+of\s+cash\s+flows?", re.I), "CashFlow"),
]

# Real FS titles are short header lines; long paragraphs that mention
# "Consolidated Balance Sheet" are notes/narrative — skip them.
_MAX_TITLE_LEN = 150


def _page_from_source(src: str) -> int | None:
    m = re.search(r"D\((\d+)", src)
    return int(m.group(1)) if m else None


def detect_fs_pages(layout_result: dict) -> dict:
    contents = layout_result["contents"][0]
    paragraphs = contents.get("paragraphs", [])
    tables = contents.get("tables", [])

    # Step 1: collect (page, statement_type) from short title paragraphs only
    hits: list[tuple[int, str]] = []
    for p in paragraphs:
        text = p.get("content", "")
        if len(text) > _MAX_TITLE_LEN:
            continue
        pg = _page_from_source(p.get("source", ""))
        if pg is None:
            continue
        for pat, stype in _FS_TITLE:
            if pat.search(text):
                hits.append((pg, stype))
                break

    if not hits:
        return {"pages": set(), "statements": {}, "content_range": "",
                "groups": []}

    # Step 2: cluster — the actual FS pages form the largest cluster (gap ≤ 3)
    all_hit_pages = sorted(set(pg for pg, _ in hits))
    clusters: list[list[int]] = []
    cur = [all_hit_pages[0]]
    for pg in all_hit_pages[1:]:
        if pg - cur[-1] <= 3:
            cur.append(pg)
        else:
            clusters.append(cur)
            cur = [pg]
    clusters.append(cur)
    fs_cluster = max(clusters, key=len)
    fs_start, fs_end = fs_cluster[0], fs_cluster[-1]

    # Step 3: extend for multi-page tables whose cells span beyond fs_end
    # Also build a table-pages lookup for per-statement extension later
    table_pages: list[set[int]] = []
    for t in tables:
        tpages = set()
        for c in t.get("cells", []):
            pg = _page_from_source(c.get("source", ""))
            if pg is not None:
                tpages.add(pg)
        table_pages.append(tpages)
        if tpages and min(tpages) >= fs_start and min(tpages) <= fs_end:
            if max(tpages) > fs_end:
                fs_end = max(tpages)

    fs_pages = set(range(fs_start, fs_end + 1))
    content_range = f"{fs_start}-{fs_end}" if fs_end > fs_start else str(fs_start)

    # Build per-statement page map (filtered to FS range)
    statements: dict[str, list[int]] = {}
    for pg, stype in hits:
        if pg in fs_pages:
            statements.setdefault(stype, [])
            if pg not in statements[stype]:
                statements[stype].append(pg)

    # Step 4: build per-statement extraction groups
    # Identify TOC/index pages — pages that match 3+ statement types
    page_type_count: dict[int, int] = {}
    for pg, _ in hits:
        if pg in fs_pages:
            page_type_count[pg] = page_type_count.get(pg, 0) + 1
    toc_pages = {pg for pg, cnt in page_type_count.items() if cnt >= 3}

    # For each statement, get its non-TOC pages and extend via table spans
    groups: list[dict] = []
    for stype, pgs in sorted(statements.items()):
        stmt_pages = sorted(pg for pg in pgs if pg not in toc_pages)
        if not stmt_pages:
            continue
        # Extend: if a table starts on a statement page and spans further, include those pages
        pg_start = min(stmt_pages)
        pg_end = max(stmt_pages)
        for tpages in table_pages:
            if tpages and min(tpages) >= pg_start and min(tpages) <= pg_end:
                pg_end = max(pg_end, max(tpages))
        # Also extend forward to the next statement's start page - 1
        # (covers pages without title paragraphs that still belong to this statement)
        all_stmt_starts = sorted(set(
            min(pg for pg in spgs if pg not in toc_pages)
            for spgs in statements.values()
            if any(pg not in toc_pages for pg in spgs)
        ))
        idx = all_stmt_starts.index(pg_start) if pg_start in all_stmt_starts else -1
        if idx >= 0 and idx + 1 < len(all_stmt_starts):
            next_start = all_stmt_starts[idx + 1]
            # Include pages between this statement and the next (e.g. continuation pages)
            pg_end = max(pg_end, next_start - 1)

        cr = f"{pg_start}-{pg_end}" if pg_end > pg_start else str(pg_start)
        groups.append({
            "statement": stype,
            "pages": list(range(pg_start, pg_end + 1)),
            "content_range": cr,
        })

    return {"pages": fs_pages, "statements": statements, "content_range": content_range,
            "groups": groups}


# --- Show page map and detection results ---

page_info = {}
for stem, result in layout_results.items():
    total = result["contents"][0].get("endPageNumber", "?")
    tables = result["contents"][0].get("tables", [])

    # Page map: what tables are on which pages
    print(f"\n{'='*60}")
    print(f"{stem}  ({total} pages, {len(tables)} tables)")
    print(f"{'='*60}")
    for i, t in enumerate(tables):
        cap = (t.get("caption") or {}).get("content", "")[:70]
        tpages = sorted(set(
            _page_from_source(c.get("source", ""))
            for c in t.get("cells", [])
            if _page_from_source(c.get("source", "")) is not None
        ))
        print(f"  table {i:2d}: pages {tpages}  {cap}")

    # Detect FS section
    info = detect_fs_pages(result)
    page_info[stem] = info
    reduction = f"{len(info['pages'])}/{total}" if isinstance(total, int) else "?"
    print(f"\n  FS section: pages {sorted(info['pages'])}  ->  content_range=\"{info['content_range']}\"  ({reduction} pages)")
    for stype, pgs in sorted(info["statements"].items()):
        print(f"    {stype:25s} {pgs}")
    print(f"\n  Extraction groups:")
    for g in info["groups"]:
        print(f"    {g['statement']:25s} pages {g['pages']}  ->  content_range=\"{g['content_range']}\"")

In [ ]:
# Quick summary of detected page ranges and extraction groups
for stem, info in page_info.items():
    total = layout_results[stem]["contents"][0].get("endPageNumber", "?")
    print(f"{stem}: content_range=\"{info['content_range']}\"  ({len(info['pages'])}/{total} pages)")
    for g in info["groups"]:
        print(f"    {g['statement']:25s} -> content_range=\"{g['content_range']}\"")
    print()

## Targeted Extraction

Run the custom analyzer **only on the detected financial-statement pages**.

In [ ]:
# Run the custom analyzer per-statement (focused extraction for better LLM accuracy)
analyzer_results = {}
for pdf in pdfs:
    info = page_info[pdf.stem]
    groups = info.get("groups", [])
    if not groups:
        print(f"  {pdf.name}: SKIP — no financial pages detected")
        continue

    print(f"  {pdf.name}:")
    merged_tables = []
    pdf_bytes = pdf.read_bytes()
    total_elapsed = 0

    for g in groups:
        cr = g["content_range"]
        stype = g["statement"]
        print(f"    {stype:25s} pages {cr} ...", end=" ", flush=True)
        t0 = time.time()
        poller = client.begin_analyze_binary(
            analyzer_id=ANALYZER_ID,
            binary_input=pdf_bytes,
            content_type="application/pdf",
            content_range=cr,
        )
        result = poller.result().as_dict()
        elapsed = time.time() - t0
        total_elapsed += elapsed

        # Collect extracted tables from this group
        tables = (
            (result.get("contents") or [{}])[0]
            .get("fields", {})
            .get("financialTables", {})
            .get("valueArray", [])
        )
        merged_tables.extend(tables)
        print(f"{len(tables)} tables in {elapsed:.1f}s")

    # Build a merged result structure
    merged_result = {
        "contents": [{
            "fields": {
                "financialTables": {
                    "type": "array",
                    "valueArray": merged_tables,
                }
            }
        }]
    }
    analyzer_results[pdf.stem] = merged_result
    print(f"    TOTAL: {len(merged_tables)} tables in {total_elapsed:.1f}s\n")

In [ ]:
# Display extracted tables summary
import sys
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from export_to_excel import load_document, _normalize_table, _array, _scalar, _obj

out_dir = REPO_ROOT / "output"
out_dir.mkdir(exist_ok=True)

for stem, result in analyzer_results.items():
    # Save raw JSON
    out_path = out_dir / f"{stem}_targeted.json"
    out_path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")

    # Normalize and display summary
    raw_tables = (
        (result.get("contents") or [{}])[0]
        .get("fields", {})
        .get("financialTables", {})
        .get("valueArray", [])
    )
    print(f"\n{'='*70}")
    print(f"{stem}: {len(raw_tables)} tables  (saved to {out_path.name})")
    print(f"{'='*70}")
    for i, raw_tbl in enumerate(raw_tables, 1):
        tobj = _obj(raw_tbl)
        title = _scalar(tobj.get("tableTitle"))
        stype = _scalar(tobj.get("statementType"))
        unit = _scalar(tobj.get("unit"))
        headers = [str(_scalar(h)) for h in _array(tobj.get("periodHeaders"))]
        rows = _array(tobj.get("rows"))
        print(f"\n  [{i}] {stype}: {title}")
        print(f"      Unit: {unit}  |  Columns: {headers}  |  Rows: {len(rows)}")
        # Show first 3 rows as preview
        for r in rows[:3]:
            robj = _obj(r)
            li = _scalar(robj.get("lineItem"))
            vals = [str(_scalar(v)) for v in _array(robj.get("values"))]
            level = _scalar(robj.get("level"), 0)
            indent = "  " * int(level)
            print(f"      {indent}{li[:50]:<50s}  {vals}")
        if len(rows) > 3:
            print(f"      ... ({len(rows) - 3} more rows)")

In [ ]:
# Export to Excel
from export_to_excel import export_document

excel_dir = out_dir / "excel"
for stem in analyzer_results:
    json_path = out_dir / f"{stem}_targeted.json"
    xlsx_path = export_document(json_path, excel_dir)
    print(f"{stem}: {xlsx_path}")